# 00 · Baseline

Baseline adalah prediksi paling sederhana yang **harus dikalahkan** oleh setiap model. Model yang tidak lebih baik dari baseline belum berguna.

| Baseline | Cara menebak | Skenario |
|---|---|---|
| **1 · Rata-rata per kelompok bidang** | Rata-rata `peminat` data latih untuk kelompok bidang yang sama | ketiga skenario |
| **2 · Peminat tahun lalu** | `prediksi = peminat_lag1` | `dengan_lag` |
| **3 · Rata-rata prodi sejenis** | `prediksi = rata_peminat_prodi_sejenis_lag1` (prodi sejenis di PTN lain, t−1) | `tanpa_riwayat_sendiri` |

Setiap baseline dijalankan untuk data gabungan (SNBP+SNBT), SNBP saja, dan SNBT saja. Semua hasil dicatat ke `hasil/evaluasi.csv` lewat `evaluasi()` dengan `versi_fondasi = 2`.

Bagian terakhir mengukur pengaruh penanda `is_rezim_vokasi_snbt` (K-19) pada prodi vokasi SNBT. Hasilnya tidak masuk `evaluasi.csv`, tetapi disimpan di `hasil/analisis_penanda_rezim.csv`.

In [1]:
import sys
from pathlib import Path

# Cari folder akar proyek (yang berisi folder src/), lalu daftarkan src/
AKAR = Path.cwd()
while not (AKAR / "src" / "fondasi.py").exists():
    if AKAR == AKAR.parent:
        raise FileNotFoundError("Folder proyek (berisi src/fondasi.py) tidak ditemukan. "
                                "Buka notebook dari dalam folder repo.")
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "src"))

import numpy as np
import pandas as pd
from fondasi import (muat_data, evaluasi, info_baris, lipatan_uji, SKENARIO,
                     FILE_EVALUASI, VERSI_FONDASI, _panel_pemodelan)

DATA = {(sk, jl): muat_data(sk, jl, verbose=False)
        for sk in SKENARIO for jl in (None, "SNBP", "SNBT")}


def prediksi_rata(skenario, X_train, y_train, X_test, kunci_latih, kunci_uji, cadangan=None):
    """Rata-rata y latih per kunci, diterapkan ke data uji menurut lipatan_uji().

    Pada skenario prodi baru, prodi uji tidak ikut menghitung rata-rata yang
    dipakai untuk menebaknya (K-24). cadangan = (kunci_latih, kunci_uji) yang
    lebih kasar, dipakai bila kunci utama tidak ada di data latih lipatan itu.
    """
    prediksi = np.full(len(X_test), np.nan)
    n_cadangan = 0
    for i_latih, i_uji in lipatan_uji(X_train, X_test, skenario):
        rata = y_train.iloc[i_latih].groupby(kunci_latih[i_latih]).mean()
        p = pd.Series(kunci_uji[i_uji]).map(rata).to_numpy(dtype=float, copy=True)
        if cadangan is not None and np.isnan(p).any():
            kosong = np.isnan(p)
            rata2 = y_train.iloc[i_latih].groupby(cadangan[0][i_latih]).mean()
            p[kosong] = pd.Series(cadangan[1][i_uji][kosong]).map(rata2).values
            n_cadangan += int(kosong.sum())
        prediksi[i_uji] = p
    assert not np.isnan(prediksi).any(), "Ada kunci uji yang tidak ada di data latih"
    if n_cadangan:
        print(f"  {n_cadangan} baris uji memakai rata-rata cadangan (kunci utama tidak ada di data latih)")
    return prediksi

## Ringkasan skenario

Jumlah baris latih dan uji, serta porsi baris yang `rata_peminat_prodi_sejenis_lag1`-nya kosong **sebelum** baris itu dikeluarkan. Nilai kosong tidak diisi (K-05); barisnya keluar dari skenario yang memakai fitur tersebut.

In [2]:
panel = _panel_pemodelan()
dasar = {
    # baris yang lolos syarat lain skenario itu, sebelum syarat prodi sejenis
    "dengan_lag": panel[panel["peminat_lag1"].notna() & panel["daya_tampung_lag1"].notna()],
    "tanpa_lag": panel,
    "tanpa_riwayat_sendiri": panel,
}
ringkas = []
for (sk, jl), (X_train, X_test, *_ ) in DATA.items():
    d = dasar[sk] if jl is None else dasar[sk][dasar[sk]["jalur"] == jl]
    kosong = d["rata_peminat_prodi_sejenis_lag1"].isna().mean()
    ringkas.append({"skenario": sk, "jalur": jl or "gabungan",
                    "baris_latih": len(X_train), "baris_uji": len(X_test),
                    "jumlah_fitur": X_train.shape[1],
                    "rata_sejenis_kosong_%": round(100 * kosong, 1),
                    "fitur_sejenis_dipakai": sk != "tanpa_lag"})
pd.DataFrame(ringkas)

,skenario,jalur,baris_latih,baris_uji,jumlah_fitur,rata_sejenis_kosong_%,fitur_sejenis_dipakai
0,dengan_lag,gabungan,21507,8508,76,8.6,True
1,dengan_lag,SNBP,10758,4254,73,8.6,True
2,dengan_lag,SNBT,10749,4254,74,8.7,True
3,tanpa_lag,gabungan,32959,9799,71,26.8,False
4,tanpa_lag,SNBP,16480,4890,68,26.9,False
5,tanpa_lag,SNBT,16479,4909,69,26.8,False
6,tanpa_riwayat_sendiri,gabungan,22453,8834,73,26.8,True
7,tanpa_riwayat_sendiri,SNBP,11216,4409,70,26.9,True
8,tanpa_riwayat_sendiri,SNBT,11237,4425,71,26.8,True


In [3]:
# Daftar fitur per skenario (data gabungan). One-hot diringkas per kelompok.
ONEHOT = ("jenjang", "portofolio", "kelompok_bidang", "provinsi", "jalur")
for sk in SKENARIO:
    kolom = DATA[(sk, None)][0].columns
    biasa = [c for c in kolom if not c.startswith(tuple(p + "_" for p in ONEHOT))]
    onehot = {p: sum(c.startswith(p + "_") for c in kolom) for p in ONEHOT}
    print(f"{sk} ({len(kolom)} fitur)")
    print("   ", ", ".join(biasa))
    print("    one-hot:", ", ".join(f"{p} ({n})" for p, n in onehot.items()))

dengan_lag (76 fitur)
    daya_tampung, jumlah_prodi_ptn, is_ptnbh, is_akademik, is_vokasi, is_ptkin, is_pendidikan, is_psdku, is_lintas_rezim_vokasi_snbt, peminat_lag1, daya_tampung_lag1, rata_peminat_prodi_sejenis_lag1, jumlah_ptn_punya_prodi_sejenis_lag1, rasio_peminat_vs_sejenis_lag1
    one-hot: jenjang (3), portofolio (12), kelompok_bidang (10), provinsi (35), jalur (2)
tanpa_lag (71 fitur)
    daya_tampung, jumlah_prodi_ptn, is_ptnbh, is_akademik, is_vokasi, is_ptkin, is_pendidikan, is_psdku, is_rezim_vokasi_snbt
    one-hot: jenjang (3), portofolio (12), kelompok_bidang (10), provinsi (35), jalur (2)
tanpa_riwayat_sendiri (73 fitur)
    daya_tampung, jumlah_prodi_ptn, is_ptnbh, is_akademik, is_vokasi, is_ptkin, is_pendidikan, is_psdku, is_lintas_rezim_vokasi_snbt, rata_peminat_prodi_sejenis_lag1, jumlah_ptn_punya_prodi_sejenis_lag1
    one-hot: jenjang (3), portofolio (12), kelompok_bidang (10), provinsi (35), jalur (2)


## Baseline 1 · Rata-rata peminat per kelompok bidang

Rata-rata dihitung **hanya dari data latih**, lalu diterapkan ke data uji 2025. Pada `tanpa_lag` dan `tanpa_riwayat_sendiri`, rata-rata dihitung per lipatan kelompok prodi sehingga prodi uji tidak ikut menghitung rata-ratanya sendiri, sama seperti model (K-24).

In [4]:
for skenario in SKENARIO:
    for jalur in (None, "SNBP", "SNBT"):
        X_train, X_test, y_train, y_test = DATA[(skenario, jalur)]
        prediksi = prediksi_rata(skenario, X_train, y_train, X_test,
                                 info_baris(X_train)["kelompok_bidang"].values,
                                 info_baris(X_test)["kelompok_bidang"].values)
        evaluasi(y_test, prediksi, "baseline_rata_kelompok", skenario, jalur,
                 n_latih=len(X_train), n_fitur=1)
        print()

[evaluasi] baseline_rata_kelompok | dengan_lag | gabungan | target=asli
  MAE=304.02  MAPE=570.51% (0 baris peminat=0 dikecualikan)  R2=0.1468  RMSLE=1.4271



[evaluasi] baseline_rata_kelompok | dengan_lag | SNBP | target=asli
  MAE=227.32  MAPE=565.65% (0 baris peminat=0 dikecualikan)  R2=0.1419  RMSLE=1.4029



[evaluasi] baseline_rata_kelompok | dengan_lag | SNBT | target=asli
  MAE=362.78  MAPE=511.43% (0 baris peminat=0 dikecualikan)  R2=0.1771  RMSLE=1.3981



[evaluasi] baseline_rata_kelompok | tanpa_lag | gabungan | target=asli
  MAE=299.57  MAPE=610.13% (0 baris peminat=0 dikecualikan)  R2=0.1296  RMSLE=1.4581



[evaluasi] baseline_rata_kelompok | tanpa_lag | SNBP | target=asli
  MAE=215.91  MAPE=560.13% (0 baris peminat=0 dikecualikan)  R2=0.1343  RMSLE=1.4200



[evaluasi] baseline_rata_kelompok | tanpa_lag | SNBT | target=asli
  MAE=364.48  MAPE=614.16% (0 baris peminat=0 dikecualikan)  R2=0.1529  RMSLE=1.4404



[evaluasi] baseline_rata_kelompok | tanpa_riwayat_sendiri | gabungan | target=asli
  MAE=302.13  MAPE=617.79% (0 baris peminat=0 dikecualikan)  R2=0.1386  RMSLE=1.4534



[evaluasi] baseline_rata_kelompok | tanpa_riwayat_sendiri | SNBP | target=asli
  MAE=225.48  MAPE=564.60% (0 baris peminat=0 dikecualikan)  R2=0.1320  RMSLE=1.4150



[evaluasi] baseline_rata_kelompok | tanpa_riwayat_sendiri | SNBT | target=asli
  MAE=361.36  MAPE=624.12% (0 baris peminat=0 dikecualikan)  R2=0.1679  RMSLE=1.4410



## Baseline 2 · Peminat tahun lalu

Menebak bahwa peminat tahun ini sama dengan tahun lalu. Hanya bisa untuk skenario `dengan_lag`.

In [5]:
for jalur in (None, "SNBP", "SNBT"):
    X_train, X_test, y_train, y_test = DATA[("dengan_lag", jalur)]
    prediksi = X_test["peminat_lag1"].values
    evaluasi(y_test, prediksi, "baseline_lag1", "dengan_lag", jalur,
             n_latih=len(X_train), n_fitur=1)
    print()

[evaluasi] baseline_lag1 | dengan_lag | gabungan | target=asli
  MAE=66.11  MAPE=23.65% (0 baris peminat=0 dikecualikan)  R2=0.9480  RMSLE=0.3633

[evaluasi] baseline_lag1 | dengan_lag | SNBP | target=asli
  MAE=49.83  MAPE=23.38% (0 baris peminat=0 dikecualikan)  R2=0.9412  RMSLE=0.3461

[evaluasi] baseline_lag1 | dengan_lag | SNBT | target=asli
  MAE=82.39  MAPE=23.92% (0 baris peminat=0 dikecualikan)  R2=0.9476  RMSLE=0.3797



## Baseline 3 · Rata-rata prodi sejenis

Menebak bahwa peminat prodi sama dengan rata-rata peminat prodi sejenis (nama baku + jenjang yang sama, jalur yang sama) di PTN lain pada tahun sebelumnya. Ini pembanding utama skenario `tanpa_riwayat_sendiri`: model yang tidak mengalahkannya berarti fitur lain tidak banyak membantu.

In [6]:
for jalur in (None, "SNBP", "SNBT"):
    X_train, X_test, y_train, y_test = DATA[("tanpa_riwayat_sendiri", jalur)]
    prediksi = X_test["rata_peminat_prodi_sejenis_lag1"].values
    evaluasi(y_test, prediksi, "baseline_rata_sejenis", "tanpa_riwayat_sendiri", jalur,
             n_latih=len(X_train), n_fitur=1)
    print()

[evaluasi] baseline_rata_sejenis | tanpa_riwayat_sendiri | gabungan | target=asli
  MAE=254.52  MAPE=382.57% (0 baris peminat=0 dikecualikan)  R2=0.3087  RMSLE=1.2352

[evaluasi] baseline_rata_sejenis | tanpa_riwayat_sendiri | SNBP | target=asli
  MAE=173.90  MAPE=308.96% (0 baris peminat=0 dikecualikan)  R2=0.4217  RMSLE=1.1497



[evaluasi] baseline_rata_sejenis | tanpa_riwayat_sendiri | SNBT | target=asli
  MAE=334.86  MAPE=455.91% (0 baris peminat=0 dikecualikan)  R2=0.2424  RMSLE=1.3149



## Ringkasan baseline

Hanya baris fondasi versi ini. Bandingkan hasil di dalam skenario yang sama: baris uji tiap skenario berbeda.

In [7]:
hasil = pd.read_csv(FILE_EVALUASI)
hasil = hasil[hasil["nama_model"].str.startswith("baseline")
              & (hasil["versi_fondasi"] == VERSI_FONDASI)]
hasil[["nama_model", "skenario", "jalur", "n_latih", "n_uji", "MAE", "MAPE", "R2", "RMSLE"]] \
    .sort_values(["skenario", "jalur", "MAE"]).reset_index(drop=True)

,nama_model,skenario,jalur,n_latih,n_uji,MAE,MAPE,R2,RMSLE
0,baseline_lag1,dengan_lag,SNBP,10758,4254,49.83,23.38,0.9412,0.3461
1,baseline_rata_kelompok,dengan_lag,SNBP,10758,4254,227.32,565.65,0.1419,1.4029
2,baseline_lag1,dengan_lag,SNBT,10749,4254,82.39,23.92,0.9476,0.3797
3,baseline_rata_kelompok,dengan_lag,SNBT,10749,4254,362.78,511.43,0.1771,1.3981
4,baseline_lag1,dengan_lag,gabungan,21507,8508,66.11,23.65,0.9480,0.3633
5,baseline_rata_kelompok,dengan_lag,gabungan,21507,8508,304.02,570.51,0.1468,1.4271
6,baseline_rata_kelompok,tanpa_lag,SNBP,16480,4890,215.91,560.13,0.1343,1.4200
7,baseline_rata_kelompok,tanpa_lag,SNBT,16479,4909,364.48,614.16,0.1529,1.4404
8,baseline_rata_kelompok,tanpa_lag,gabungan,32959,9799,299.57,610.13,0.1296,1.4581
9,baseline_rata_sejenis,tanpa_riwayat_sendiri,SNBP,11216,4409,173.90,308.96,0.4217,1.1497


## Pengaruh penanda vokasi SNBT (K-19)

Peminat prodi vokasi (D3/D4) jalur SNBT melonjak mulai 2024; penyebabnya belum diketahui, jadi penanda hanya menandai diskontinuitasnya. Ada dua penanda:

| Penanda | Bernilai 1 untuk SNBT D3/D4 | Dipakai di |
|---|---|---|
| `is_rezim_vokasi_snbt` | tahun ≥ 2024 (sudah dalam rezim baru) | `tanpa_lag` |
| `is_lintas_rezim_vokasi_snbt` | tahun = 2024 (pembanding t−1 masih dari sebelum lonjakan) | `dengan_lag`, `tanpa_riwayat_sendiri` |

**Pertumbuhan per tahun.** Rasio median peminat terhadap pembandingnya pada tahun sebelumnya, untuk prodi vokasi SNBT. Rasio 2024 jauh lebih besar daripada 2025, sehingga penanda tahun ≥ 2024 mempunyai arti berbeda di data latih (lonjakan) dan data uji (sudah stabil) pada skenario berbasis t−1. Karena itu skenario tersebut memakai penanda lintas rezim.

In [8]:
v = panel[(panel["jalur"] == "SNBT") & panel["jenjang"].isin(["D3", "D4"])]
pd.DataFrame([{
    "tahun": t,
    "is_rezim_vokasi_snbt": int(s["is_rezim_vokasi_snbt"].iloc[0]),
    "n_baris": len(s),
    "median_peminat": s["peminat"].median(),
    "median_peminat_per_lag1": round((s["peminat"] / s["peminat_lag1"]).median(), 2),
    "median_peminat_per_rata_sejenis": round(
        (s["peminat"] / s["rata_peminat_prodi_sejenis_lag1"]).median(), 2),
} for t, s in v.groupby("tahun")])

,tahun,is_rezim_vokasi_snbt,n_baris,median_peminat,median_peminat_per_lag1,median_peminat_per_rata_sejenis
0,2021,0,363,112.0,NaN,NaN
1,2022,0,427,112.0,1.06,0.96
2,2023,0,1104,76.0,0.90,0.71
3,2024,1,1208,264.5,3.23,2.83
4,2025,1,1278,354.5,1.30,0.91


**Baseline dengan dan tanpa penanda.** Versi *+ penanda* belajar satu angka tambahan per nilai penanda dari data latih, seperti yang bisa dilakukan model:

| Skenario | Baseline tanpa penanda | Baseline + penanda |
|---|---|---|
| `tanpa_lag` | rata-rata per kelompok bidang | rata-rata per kelompok bidang × penanda |
| `dengan_lag` | `peminat_lag1` | `peminat_lag1` × median rasio peminat/lag1 latih per nilai penanda |
| `tanpa_riwayat_sendiri` | rata-rata prodi sejenis | rata-rata prodi sejenis × median rasio latih per nilai penanda |

Pada skenario berbasis t−1, kedua penanda diuji: penanda yang dipakai (lintas rezim) dan penanda tahun ≥ 2024 yang ditolak. Dinilai pada data uji SNBT 2025: subset vokasi (D3/D4) dan seluruh SNBT.

In [9]:
def nilai(y, p):
    g = np.asarray(y) - np.asarray(p)
    return {"MAE": round(float(np.mean(np.abs(g))), 2),
            "R2": round(float(1 - np.sum(g ** 2) / np.sum((y - np.mean(y)) ** 2)), 4),
            "median_galat": round(float(np.median(-g)), 1)}   # + = menebak terlalu tinggi

baris = []
for sk in SKENARIO:
    X_train, X_test, y_train, y_test = DATA[(sk, "SNBT")]
    i_latih, i_uji = info_baris(X_train), info_baris(X_test)
    dipakai = "is_rezim_vokasi_snbt" if sk == "tanpa_lag" else "is_lintas_rezim_vokasi_snbt"
    assert dipakai in X_train.columns
    versi = {"tanpa penanda": None, f"+ {dipakai} (dipakai)": dipakai}
    if sk != "tanpa_lag":
        versi["+ is_rezim_vokasi_snbt (ditolak)"] = "is_rezim_vokasi_snbt"

    prediksi = {}
    for label, kolom in versi.items():
        if sk == "tanpa_lag":
            nama = "rata_kelompok"
            k_latih, k_uji = i_latih["kelompok_bidang"].values, i_uji["kelompok_bidang"].values
            if kolom is None:
                p = prediksi_rata(sk, X_train, y_train, X_test, k_latih, k_uji)
            else:
                p = prediksi_rata(sk, X_train, y_train, X_test,
                                  k_latih + "|" + i_latih[kolom].astype(str).values,
                                  k_uji + "|" + i_uji[kolom].astype(str).values,
                                  cadangan=(k_latih, k_uji))
        else:
            acuan = "peminat_lag1" if sk == "dengan_lag" else "rata_peminat_prodi_sejenis_lag1"
            nama = "lag1" if sk == "dengan_lag" else "rata_sejenis"
            p = X_test[acuan].values
            if kolom is not None:
                rasio = (y_train / X_train[acuan]).groupby(i_latih[kolom].values).median()
                p = p * i_uji[kolom].map(rasio).values
        assert not np.isnan(p).any(), f"{sk} {label}: nilai penanda uji tidak ada di data latih"
        prediksi[label] = p

    vok = i_uji["jenjang"].isin(["D3", "D4"]).values
    for subset, m in (("SNBT vokasi 2025", vok), ("SNBT seluruhnya 2025", np.ones_like(vok))):
        m = m.astype(bool)
        for label, p in prediksi.items():
            baris.append({"skenario": sk, "baseline": nama, "versi": label, "subset": subset,
                          "n_uji": int(m.sum()), **nilai(y_test.values[m], p[m])})

penanda = pd.DataFrame(baris)
penanda.to_csv(AKAR / "hasil" / "analisis_penanda_rezim.csv", index=False, encoding="utf-8-sig")
penanda

  1 baris uji memakai rata-rata cadangan (kunci utama tidak ada di data latih)


,skenario,baseline,versi,subset,n_uji,MAE,R2,median_galat
0,dengan_lag,lag1,tanpa penanda,SNBT vokasi 2025,902,123.83,0.9451,-51.5
1,dengan_lag,lag1,+ is_lintas_rezim_vokasi_snbt (dipakai),SNBT vokasi 2025,902,134.01,0.9410,-66.2
2,dengan_lag,lag1,+ is_rezim_vokasi_snbt (ditolak),SNBT vokasi 2025,902,1383.64,-7.8088,501.4
3,dengan_lag,lag1,tanpa penanda,SNBT seluruhnya 2025,4254,82.39,0.9476,-22.0
4,dengan_lag,lag1,+ is_lintas_rezim_vokasi_snbt (dipakai),SNBT seluruhnya 2025,4254,87.17,0.9450,-32.0
5,dengan_lag,lag1,+ is_rezim_vokasi_snbt (ditolak),SNBT seluruhnya 2025,4254,352.13,-2.4741,-10.7
6,tanpa_lag,rata_kelompok,tanpa penanda,SNBT vokasi 2025,1278,503.60,0.0810,48.1
7,tanpa_lag,rata_kelompok,+ is_rezim_vokasi_snbt (dipakai),SNBT vokasi 2025,1278,514.82,0.1682,134.5
8,tanpa_lag,rata_kelompok,tanpa penanda,SNBT seluruhnya 2025,4909,364.48,0.1529,86.9
9,tanpa_lag,rata_kelompok,+ is_rezim_vokasi_snbt (dipakai),SNBT seluruhnya 2025,4909,365.50,0.1917,92.7


Cara membaca: bila versi *+ penanda* menurunkan MAE pada subset vokasi, penanda membantu di skenario itu. Bila MAE-nya naik dan `median_galat` positif besar, penanda membuat tebakan terlalu tinggi karena pola yang dipelajari dari 2024 tidak berlaku untuk 2025.

Baseline ini hanya bisa memakai penanda sebagai pengali. Model pohon juga dapat mempelajari lonjakan 2024 lewat kombinasi `jalur` × `jenjang` bila tidak ada penanda yang menyerapnya; itulah alasan penanda lintas rezim tetap disertakan walaupun pada data uji nilainya selalu 0. Keputusan dan alasannya ada di `docs/KEPUTUSAN_METODOLOGI.md` (K-19).